## Installations
---

In [ ]:
%%capture

!pip install -q gdown segmentation-models-pytorch albumentations
!pip install -q transformers torchgeo grad-cam

### Import Libraries
---

In [ ]:
import os
import json
import random
import zipfile

from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
import Albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp
from transformers import SegformerFeatureExtractor, SegformerForSemanticSegmentation, SegformerConfig
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold
from tqdm.auto import tqdm

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

DEVICE

## Configurations
---

In [1]:
CFG = {
    "arch" : "segformer",
    "init" : "raw",

    "run_version" : "v1",
    "img_size" : 512,
    "batch_size" : 4,
    "epochs" : 40,
    "patience" : 8,
    "lr_encoder" : 1e-5,
    "lr_decoder" : 3e-4,
    "weight_decay" : 1e-4,
    "seed" : 42,
    "dice_weight" : 0.7,
    "ce_weight" : 0.3,
    "threshold" : 0.5,
    "dup_hamming" : 5
}

In [2]:
CFG['run_id'] = f"{CFG['arch']}-{CFG['init']}"

CFG

{'arch': 'segformer',
 'init': 'raw',
 'run_version': 'v1',
 'img_size': 512,
 'batch_size': 4,
 'epochs': 40,
 'patience': 8,
 'lr_encoder': 1e-05,
 'lr_decoder': 0.0003,
 'weight_decay': 0.0001,
 'seed': 42,
 'dice_weight': 0.7,
 'ce_weight': 0.3,
 'threshold': 0.5,
 'dup_hamming': 5,
 'run_id': 'segformer-raw'}

- Seeding (Randomness)

In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

In [ ]:
set_seed(CFG['seed'])

NameError: name 'np' is not defined

## Download Dataset
---

In [11]:
file_id = "114xnPVZtthHi4Bejzho33vGJSqb4-F8g"
output_zip = "/content/dataset.zip"
extract_path = Path("/content/flopwd_data")

In [ ]:
!gdown --id {file_id} -O {output_zip}

if os.path.exists(output_zip):
    os.makedirs(extract_path, exist_ok=True)
    with zipfile.ZipFile(output_zip, "r") as zip_ref:
        zip_ref.extractall(extract_path)
    print(f"Success! Dataset extracted to {extract_path}")

## Import Dataset
---

In [12]:
dataset_path = extract_path / "Dal Lake Floating Plastic Waste Detection Dataset (FloPWD 2025)"
raw_dir = dataset_path / "Raw_Images"
mask_dir = dataset_path / "Segmentation_Masks"
labels_csv = dataset_path / "Image_labels_Binary Classification Task.csv"
reg_csv = dataset_path / "Mask_foreground_percentages_Regression Task.csv"

In [ ]:
"images:", len(list(raw_dir.glob("*.jpg")))

In [ ]:
"masks :", len(list(mask_dir.glob("*.png")))

### Build Dataframe

In [ ]:
labels_df = pd.read_csv(labels_csv)

labels_df = labels_df.rename(columns={
    "image name" : "image_name",
    "Presence of plastic waste?" : "presence"
})

In [ ]:
reg_df = pd.read_csv(reg_csv)

reg_df = reg_df.rename(columns={
    "image name" : "image_name",
    "plastic waste accumulation (in percentage)" : "fg_pct"
})

- merge

In [ ]:
df = labels_df.merge(reg_df, on="image_name", how="inner")

df['image_path'] = df['image_name'].apply(lambda x: raw_dir / x)
df['mask_path'] = df['image_name'].str.replace('.jpg', "_mask.png", regex=False).apply(lambda x: str(mask_dir / x))
df['has_plastic'] = (df['presence'].astype(str).str.lower() == 'yes').astype(int)
df['fg_pct'] = df['fg_pct'].astype(float)

In [ ]:
df.head()

### EDA
---

In [ ]:
r = df.iloc[0]
img = cv2.cvtColor(cv2.imread(r['image_path']), cv2.COLOR_BGR2RGB)
mask = (cv2.imread(r['mask_path'], cv2.IMREAD_GRAYSCALE) > 127).astype(np.uint8)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))

ax[0].imshow(img)
ax[0].set_title(r['image_name'])

ax[1].imshow(mask, cmap='gray')
ax[1].set_title(f'mask - {mask.mean()*100:.2f}%')

ax[2].imshow(img)
ax[2].imshow(mask, alpha=0.4, cmap='autumn')
ax[2].set_title(f'overlay - {mask.mean()*100:.2f}%')

for a in ax:
    a.axis('off')

## Split
---

In [ ]:
train_val_df, test_df = train_test_split(df, test_size=0.15, random_state=CFG['seed'], stratify=df['has_plastic'])

In [ ]:
train_df, val_df = train_test_split(train_val_df, test_size=15/85, random_state=CFG['seed'], stratify=train_val_df['has_plastic'])

In [ ]:
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [ ]:
len(train_df), len(val_df), len(test_df)

## Image Augmentation
---

In [ ]:
S = CFG['img_size']
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

In [ ]:
train_df = A.Compose([
    A.Resize(S, S),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=30, p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.4),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2()
])

In [ ]:
eval_tf = A.Compose([
    A.Resize(S, S), 
    A.Normalize(mean=MEAN, std=STD), 
    ToTensorV2()
])

## Training - Dataset
---

In [ ]:
class FloPWD(dataset):

    def __init__(self, frame, tf):
        self.df, self.tf = frame.reset_index(drop=True), tf
    
    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = cv2.cvtColor(cv2.imread(r['image_path']), cv2.COLOR_BGR2RGB)
        mask = (cv2.imread(r['mask_path'], cv2.IMREAD_GRAYSCALE) > 127).astype(np.uint8)
        out = self.tf(image=img, mask=mask)
        return out['image'], out['mask'].long()

## Training - DataLoader
---

In [ ]:
B = CFG['batch_size']

In [ ]:
train_loader = DataLoader(FloPWD(train_df, train_tf), batch_size=B, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(FloPWD(val_df, eval_tf), batch_size=B, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(FloPWD(test_df, eval_tf), batch_size=B, shuffle=False, num_workers=2, pin_memory=True)

## Model
---

In [ ]:
ARCH = {
    "unet" : smp.Unet,
    "deeplab" : smp.DeepLabV3Plus,
    "unetpp" : smp.UnetPlusPlus,
    "upernet" : smp.UPerNet
}

In [ ]:
def build_model(arch, init):
    if arch in ARCH:
        enc = "tu-convnextv2_nano" if arch == "upernet" else "resnet50"
        model = ARCH[arch](
            encoder_name = enc,
            encoder_weights = "imagenet" if init == "imagenet" else None,
            classes = 2
        )

        if init == "aerial":
            from torchgeo.models import ResNet50_Weights
            sd = ResNet50_Weights.FMOW_RGB_GASSL.get_state_dict(progress=True)
            missing, unexpected = model.encoder.load_state_dict(sd, strict=False)

        return model

    src = {
        "raw" : None,
        "imagenet" : "nvidia/mit-b3",
        "aerial" : "chribark/segformer-b3-finetuned-UAVid"
    }

    src = src[init]

    if src is None:
        cfg = SegformerConfig.from_pretrained("nvidia/mit-b3", num_labels=2)

        return SegformerForSemanticSegmentation(cfg)

    return SegformerForSemanticSegmentation.from_pretrained(src, num_labels=2, ignore_mismatched_sizes=True)

In [ ]:
model = build_model(CFG['arch'], CFG['init']).to(DEVICE)

### Weight Fingerprint
---

In [ ]:
p = next(model.parameters())

In [ ]:
f"{CFG['run_id']:24s} mean={float(p.mean()):+.6f}  std={float(p.std()):.6f}"

In [ ]:
"parameters:", round(sum(q.numel() for q in model.parameters()) / 1e6, 2), "M"

## Loss
---

In [ ]:
sub = train_df.sample(
    min(200, len(train_df)),
    random_state=CFG['seed']
)

pos = 0
neg = 0

In [ ]:
for _, r in sub.iterrows():
    mask = cv2.imread(r["mask_path"], cv2.IMREAD_GRAYSCALE) > 127
    pos += int(mask.sum())
    neg += int(mask.size - mask.sum())

In [ ]:
pos_weight = neg / max(pos, 1)

pos_weight

### Dice Loss 
---

In [ ]:
def dice_loss(logits, targets, smooth=1.0):
    probs = torch.softmax(logits, dim=1)[:, 1]
    t = targets.float()
    inter = (probs * t).sum(dim=(1, 2))
    union = probs.sum(dim=(1, 2)) + t.sum(dim=(1, 2))
    return 1.0 - ((2 * inter + smooth) / (union + smooth)).mean()

### Criterion
---

In [ ]:
w  = torch.tensor([1.0, float(min(pos_weight, 50.0))], device=DEVICE)
ce = nn.CrossEntropyLoss(weight=w)

In [ ]:
def criterion(logits, targets):
    return CFG["dice_weight"] * dice_loss(logits, targets) + CFG["ce_weight"] * ce(logits, targets)

## Optimizer
---

In [ ]:
if CFG['arch'] == 'segformer':
    enc = list(model.segformer.parameters())
    dec = list(model.decode_head.parameters())
else:
    enc = list(model.encoder.parameters())
    dec = [q for n, q in model.named_parameters() if not n.startswith('encoder.')]

In [ ]:
lr_enc = CFG["lr_decoder"] if CFG["init"] == "raw" else CFG["lr_encoder"]

In [ ]:
optimizer = torch.optim.AdamW(
    [{"params": enc, "lr": lr_enc}, {"params": dec, "lr": CFG["lr_decoder"]}],
    weight_decay=CFG["weight_decay"
])

In [ ]:
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))

## Training
---

### Forward Pass

In [ ]:
def forward_logits(model, x):
    out = model(x)

    if hasattr(out, "logits"):
        return F.interpolate(out.logits, size=x.shape[-2:], mode='bilinear', align_corners=False)

    return out

### Metrics

In [ ]:
def compute_stats(preds, targets, eps=1e-7):
    p, t = preds.float(), targets.float()
    tp = (p * t).sum(dim=(1, 2))
    fp = (p * (1 - t)).sum(dim=(1, 2))
    fn = ((1 - p) * t).sum(dim=(1, 2))
    tn = ((1 - p) * (1 - t)).sum(dim=(1, 2))
    fg = (tp + eps) / (tp + fp + fn + eps)
    bg = (tn + eps) / (tn + fp + fn + eps)

    return {"foreground_iou": fg, "background_iou": bg, "miou": (fg + bg) / 2,
            "dice": (2*tp + eps) / (2*tp + fp + fn + eps),
            "precision": (tp + eps) / (tp + fp + eps),
            "recall":    (tp + eps) / (tp + fn + eps)
    }

### Epoch

In [ ]:
def train_epoch():
    model.train()

    total = 0.0
    for x, y in tqdm(train_loader, desc='train', leave=False):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda", enabled=(DEVICE == "cuda")):
            loss = criterion(forward_logits(model, x), y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total += loss.item()

    return total / len(train_loader)

In [ ]:
@torch.no_grad()
def evaluate(loader):
    model.eval()

    acc = {}
    total = 0.0
    for x, y in tqdm(loader, desc="eval", leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)

        logits = forward_logits(model, x)

        total += criterion(logits, y).item()

        pred = (torch.softmax(logits, dim=1)[:, 1] > CFG["threshold"]).float()

        for k, v in compute_stats(pred, y).items():
            acc.setdefault(k, []).append(v.cpu())

    res = {k: float(torch.cat(v).mean()) for k, v in acc.items()}

    res["loss"] = total / len(loader)
    
    return res

### Main Loop

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
OUT = Path(f"/content/drive/MyDrive/THESIS/runs/{CFG['run_id']}/{CFG['run_version']}")
OUT.mkdir(parents=True, exist_ok=True)
best_path = OUT / "best.pth"

In [ ]:
best, bad, history = -1.0, 0, []

for ep in range(1, CFG["epochs"] + 1):
    tr = train_epoch()

    va = evaluate(val_loader)

    history.append({"epoch": ep, "train_loss": tr, **{f"val_{k}": v for k, v in va.items()}})
    
    if va["miou"] > best:
        best, bad = va["miou"], 0
        torch.save(model.state_dict(), best_path)
        print(f"saved (mIoU {best:.4f})")
    else:
        bad += 1
        if bad >= CFG["patience"]:
            print(f"early stop at epoch {ep}") 
            break

In [ ]:
pd.DataFrame(history).to_csv(OUT / "history.csv", index=False)

In [ ]:
train_df[["image_name"]].assign(split="train").to_csv(OUT / "split_train.csv", index=False)
val_df[["image_name"]].assign(split="val").to_csv(OUT / "split_val.csv", index=False)
test_df[["image_name"]].assign(split="test").to_csv(OUT / "split_test.csv", index=False)

## Results
---

In [ ]:
model.load_state_dict(torch.load(best_path, map_location=DEVICE))


In [ ]:
res = evaluate(test_loader)

In [ ]:
res.update({"run_id": CFG["run_id"], "version": CFG["run_version"],
            "best_val_miou": best, "n_train": len(train_df),
            "n_val": len(val_df), "n_test": len(test_df), "cfg": CFG})

In [ ]:
with open(OUT / "test_metrics.json", "w") as f:
    json.dump(res, f, indent=2)
    
print(json.dumps({k: round(v, 4) for k, v in res.items() if isinstance(v, float)}, indent=2))

### Samples

In [ ]:
SAMPLES = ["img1484.jpg", "img324.jpg", "img2.jpg", "img563.jpg"]
rows = [df[df.image_name == n].iloc[0] for n in SAMPLES]

fig, ax = plt.subplots(len(rows), 3, figsize=(13, 4 * len(rows)))

for i, r in enumerate(rows):
    img = cv2.cvtColor(cv2.imread(r["image_path"]), cv2.COLOR_BGR2RGB)

    gt  = (cv2.imread(r["mask_path"], cv2.IMREAD_GRAYSCALE) > 127).astype(np.uint8)

    x   = eval_tf(image=img, mask=gt)["image"][None].to(DEVICE)

    with torch.no_grad():
        pr = (torch.softmax(forward_logits(model, x), 1)[0, 1] > CFG["threshold"]).cpu().numpy()

    ax[i,0].imshow(img)            
    ax[i,0].set_title(r["image_name"])
    ax[i,1].imshow(gt, cmap="gray")
    ax[i,1].set_title(f"truth {gt.mean()*100:.2f}%")
    ax[i,2].imshow(pr, cmap="gray") 
    ax[i,2].set_title(f"pred {pr.mean()*100:.2f}%")

    for a in ax[i]: 
        a.axis("off")

plt.tight_layout()
plt.savefig(OUT / "samples.png", dpi=110)